# Vision Transformer in PyTorch (Simple Version)

This notebook shows a simplified PyTorch version of the Vision Transformer idea.

The main goal is to understand the full workflow in a simpler way:
- use a CNN to extract image features,
- convert those features into tokens,
- add position information,
- pass tokens through a transformer,
- and classify the result.

This version keeps the code easier to read and the explanations shorter.

This cell imports the libraries needed to build the model, process images, and train the network. It includes PyTorch, the dataset loader, and image transformation tools.


## 1. Import libraries

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print('PyTorch ready')

This cell fixes the random seed so training is repeatable. Without this, each run may start differently and produce different results.


## 2. Set random seed

This cell fixes the random seed so that training is repeatable. Without this, each run may start differently and produce different results.

In [ ]:
seed = 7331
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

This CNN extracts feature maps from input images. It learns local patterns such as edges, textures, and shapes before the transformer processes the data.


## 3. Define a CNN backbone

This CNN extracts feature maps from input images. It learns local patterns such as edges, textures, and shapes before the transformer sees the data.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(64),
            nn.Conv2d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(128),
            nn.Conv2d(128, 256, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(256),
            nn.Conv2d(256, 512, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(512),
            nn.Conv2d(512, 1024, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(1024)
        )

    def forward(self, x):
        return self.features(x)

This cell converts the CNN feature map into tokens. Transformers work with a sequence of tokens, so the feature map is flattened before attention is applied.


## 4. Patch embedding for transformer tokens

This cell converts the CNN feature map into tokens. Transformers expect a sequence, so the feature map is flattened into a token list so self-attention can work on it.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, in_ch=1024, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=1)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        return x

This cell defines the attention mechanism. It allows the model to decide which parts of the image are most important when making a prediction.


## 5. Multi-head self-attention block

This cell defines attention, which lets the model decide which parts of the image are important. Multi-head attention helps the model look at different relationships at the same time.

In [ ]:
class MHSA(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.0):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.reshape(B, N, self.heads, -1).transpose(1, 2)
        k = k.reshape(B, N, self.heads, -1).transpose(1, 2)
        v = v.reshape(B, N, self.heads, -1).transpose(1, 2)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(x))

## 6. Transformer block

This cell defines a transformer block, which is the main building unit of a Vision Transformer. It combines attention and a small feed-forward network to learn useful patterns from the tokens.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MHSA(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

## 7. Build the hybrid model

This cell creates the complete hybrid CNN + Vision Transformer model. The CNN extracts features, the transformer processes them, and the final layer predicts the class.

In [ ]:
class ViT(nn.Module):
    def __init__(self, in_ch=1024, num_classes=2, embed_dim=768, depth=6, heads=8):
        super().__init__()
        self.patch = PatchEmbed(in_ch, embed_dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos = nn.Parameter(torch.randn(1, 50, embed_dim))
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch(x)
        B, L, _ = x.shape
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat((cls, x), dim=1)
        x = x + self.pos[:, :L + 1]
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x)[:, 0])

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.cnn = ConvNet()
        self.vit = ViT(num_classes=num_classes)

    def forward(self, x):
        features = self.cnn(x)
        return self.vit(features)

model = CNN_ViT_Hybrid(num_classes=2)
print(model)

## 8. Prepare dataset

This cell loads the image dataset and prepares the training and validation sets. It also applies image transformations like resizing, rotation, and normalization so the model sees clean and consistent data.

In [ ]:
DATASET_PATH = os.path.join('.', 'images_dataSAT')

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(40),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

## 9. Train the model

This cell performs the actual training. It loops over the dataset, computes the loss, backpropagates the error, and updates the model weights using the optimizer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'Epoch {epoch + 1} - loss: {running_loss / len(train_loader):.4f}')

## 10. Evaluate the model

This final code cell checks how well the trained model performs on unseen validation images. It computes the validation accuracy to measure how good the model is.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'Validation accuracy: {correct / total:.4f}')

## Summary

This notebook shows the main idea behind a CNN-ViT hybrid model in PyTorch:
- CNN extracts local image features,
- ViT uses attention to model global relationships,
- and the final layer predicts the class.

This combination is powerful because it uses both local and global information together.